In [1]:
from surprise import Dataset
from surprise import Reader
from surprise import SVD

from surprise.model_selection import train_test_split
from surprise.accuracy import rmse


In [2]:
import pandas as pd
mv=pd.read_excel(r"C:\Users\RohanS\Downloads\movie.xlsx")
rt=pd.read_csv(r"C:\Users\RohanS\Downloads\ratings.csv")


In [3]:
ratings_sample = rt.sample(
    n=500000,
    random_state=42
)
ratings_sample.head()

,userId,movieId,rating,timestamp
17679788,122270,8360,3.5,1335056824
7106385,49018,32,2.0,1000194636
12970708,89527,109374,3.5,1420536400
15426752,106704,1060,3.0,948576477
6934678,47791,1732,2.0,1137685703


In [4]:
reader = Reader(rating_scale=(0.5,5))
data = Dataset.load_from_df(
    ratings_sample[['userId','movieId','rating']],
    reader
)

In [5]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)


In [6]:
model = SVD()
model.fit(trainset)
predictions = model.test(testset)
rmse(predictions)

RMSE: 0.9205


np.float64(0.9204932080983812)

In [7]:
model.predict(
    uid=1,
    iid=109487
)

Prediction(uid=1, iid=109487, r_ui=None, est=np.float64(4.030242919498779), details={'was_impossible': False})

In [9]:
movie_lookup = mv[
    ['movieId','title']
]

In [20]:
def recommend_svd(user_id, n=10):

    movie_ids = mv['movieId'].unique()

    predictions = []

    for movie_id in movie_ids:

        pred = model.predict(
            user_id,
            movie_id
        )

        predictions.append(
            (movie_id, pred.est)
        )

    predictions = sorted(
        predictions,
        key=lambda x:x[1],
        reverse=True
    )

    top_movies = predictions[:n]

    rec_df = pd.DataFrame(
        top_movies,
        columns=['movieId','pred_rating']
    )

    return rec_df.merge(
        movie_lookup,
        on='movieId'
    )
recommend_svd(5545)

,movieId,pred_rating,title
0,913,4.312406,"Maltese Falcon, The"
1,4993,4.200141,"Lord of the Rings: The Fellowship of the Ring,..."
2,4973,4.160300,"Amelie (Fabuleux destin d'Amélie Poulain, Le)"
3,668,4.125859,Song of the Little Road (Pather Panchali)
4,1945,4.112398,On the Waterfront
5,912,4.083167,Casablanca
6,318,4.068373,"Shawshank Redemption, The"
7,923,4.056086,Citizen Kane
8,2858,4.034619,American Beauty
9,3468,4.032970,"Hustler, The"


In [25]:
movie_ratings = ratings_sample.merge(
    mv[['movieId','title','genres']],
    on='movieId'
)

movie_stats = movie_ratings.groupby(
    'title'
).agg(
    avg_rating=('rating','mean'),
    num_ratings=('rating','count'),
    genres=('genres','first')
).reset_index()
movie_stats=movie_stats.sort_values('num_ratings',ascending=False)
import numpy as np
movie_stats['score'] = (
    movie_stats['avg_rating']
    *
    np.log1p(movie_stats['num_ratings'])/10)
def recommend_popular(n):
    return movie_stats[
        ['title','avg_rating','num_ratings','score','genres']
    ].head(n)
recommend_popular(150)

,title,avg_rating,num_ratings,score,genres
9080,Pulp Fiction,4.136311,1713,3.080139,Comedy|Crime|Drama|Thriller
4184,Forrest Gump,4.027812,1636,2.980831,Comedy|Drama|Romance|War
10063,"Shawshank Redemption, The",4.496873,1599,3.317685,Crime|Drama
10176,"Silence of the Lambs, The",4.165926,1576,3.067488,Crime|Horror|Thriller
6110,Jurassic Park,3.676510,1490,2.686500,Action|Adventure|Sci-Fi|Thriller
...,...,...,...,...,...
1123,"Beautiful Mind, A",3.931947,529,2.466462,Drama|Romance
4790,"Green Mile, The",3.964015,528,2.485829,Crime|Drama
1794,Broken Arrow,3.152462,528,1.976905,Action|Adventure|Thriller
8282,Office Space,3.980952,525,2.494187,Comedy|Crime


In [62]:
import random
l=[]
k=[]
def hybrid_recommend(user_id=None):
    if user_id==None:
        for i in recommend_popular(1000)['title']:
            k.append(i)
        
        for j in range(10):
            x=random.randint(0,149)
            l.append(k[x])
        return l
    else:
       print( recommend_svd(user_id))
            
hybrid_recommend()

['Psycho ',
 'Twister ',
 'Apocalypse Now ',
 'E.T. the Extra-Terrestrial ',
 'Die Hard ',
 'Pretty Woman ',
 'Die Hard ',
 'Total Recall ',
 'Jerry Maguire ',
 'Sleepless in Seattle ']